# Assignment No.8 — MSL Recognition Tutorial (Class-25)

**Student:** Yee Mon Thant

**Course:** AIE-F

**Instructor:** Ye Kyaw Thu, Lab. Leader, Language Understanding Lab., Myanmar

**Class Repo:** [https://github.com/ye-kyaw-thu/AIE-F/tree/main/slide-code/class-25](https://github.com/ye-kyaw-thu/AIE-F/tree/main/slide-code/class-25)

**Corpus Link:** [https://github.com/ye-kyaw-thu/MSL4Emergency](https://github.com/ye-kyaw-thu/MSL4Emergency)

---

**Acknowledgement:** All Python scripts, shell scripts, and configuration files used in this notebook (`src/`, `scripts/`, `config/`) were provided by Sayar Ye Kyaw Thu as part of Class-25. Only the required assignment change — using MSL gloss (annotations.txt column 2) instead of the default normal Myanmar text (column 1) as the classification label in `src/utils.py` — was made by the student.

This notebook implements Myanmar Sign Language video recognition using MSL gloss labels. Three models — BiLSTM, Transformer, and ST-GCN — are trained and compared on the MSL4Emergency dataset.

---

## 1. Clone repos and install dependencies
Clone the teacher's AIE-F repo and the MSL4Emergency video dataset repo, then install required Python packages.

In [1]:
!git clone https://github.com/ye-kyaw-thu/AIE-F.git
!git clone https://github.com/ye-kyaw-thu/MSL4Emergency.git

%cd AIE-F/slide-code/class-25/msl_recognition
!pip install -r requirements.txt

fatal: destination path 'AIE-F' already exists and is not an empty directory.
fatal: destination path 'MSL4Emergency' already exists and is not an empty directory.
/kaggle/working/AIE-F/slide-code/class-25/msl_recognition
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.5/36.5 MB 43.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 9.9 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


## 2. Confirm GPU is available
Check that PyTorch can see the Kaggle GPU (should print True and the device name, e.g. Tesla T4).

In [2]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True Tesla T4


## 3. Free up disk space
Remove .git history from both cloned repos (not needed after cloning) to avoid running out of disk space.

In [3]:
!rm -rf /kaggle/working/MSL4Emergency/.git
!rm -rf /kaggle/working/AIE-F/.git
!df -h /kaggle/working

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   15G  4.7G  77% /kaggle/working


## 4. Link videos into data/videos

In [23]:
!rm -rf data/videos
!mkdir -p data/videos
!find /kaggle/working/MSL4Emergency/msl4emergency-ver-1.0/video -name "*.mp4" -exec ln -sf {} data/videos/ \;
!ls data/videos | wc -l

558


## 5. Assignment change: use column 2 (MSL gloss) as the label
Find where utils.py sets the classification label, to prepare for the column-2 (MSL gloss) change. Edit utils.py so the model classifies videos by MSL gloss instead of the full Myanmar sentence. Then, fix the comment in utils.py so the documentation reflects the column-2 label change.

In [5]:
!grep -n "'label'" src/utils.py

75:        List of dicts with keys: 'idx', 'normal_text', 'msl_gloss', 'label'
76:        where 'label' is the msl_gloss (assignment: column 2, used as class identifier).
105:                'label':       msl_gloss,     # class = MSL gloss (assignment: column 2)
123:        lbl = rec['label']
251:    labels  = [label2idx[r['label']] for r in records]
356:    labels        = [r['label'] for r in records]
473:    labels  = [label2idx[r['label']] for r in records]


In [6]:
!sed -i "105s/normal_text,   # class = full Myanmar phrase/msl_gloss,     # class = MSL gloss (assignment: column 2)/" src/utils.py
!grep -n "'label'" src/utils.py

75:        List of dicts with keys: 'idx', 'normal_text', 'msl_gloss', 'label'
76:        where 'label' is the msl_gloss (assignment: column 2, used as class identifier).
105:                'label':       msl_gloss,     # class = MSL gloss (assignment: column 2)
123:        lbl = rec['label']
251:    labels  = [label2idx[r['label']] for r in records]
356:    labels        = [r['label'] for r in records]
473:    labels  = [label2idx[r['label']] for r in records]


In [7]:
!sed -i "76s/where 'label' is the normal_text (used as class identifier)./where 'label' is the msl_gloss (assignment: column 2, used as class identifier)./" src/utils.py
!sed -n '70,110p' src/utils.py


    Format (tab-delimited):
        normal_myanmar_text  <TAB>  msl_gloss

    Returns:
        List of dicts with keys: 'idx', 'normal_text', 'msl_gloss', 'label'
        where 'label' is the msl_gloss (assignment: column 2, used as class identifier).
    """
    records = []
    ann_path = Path(ann_path)

    if not ann_path.exists():
        raise FileNotFoundError(f"Annotation file not found: {ann_path}")

    with open(ann_path, 'r', encoding='utf-8') as f:
        for line_no, line in enumerate(f):
            line = line.rstrip('\n')
            if not line.strip():
                continue

            parts = line.split('\t')
            if len(parts) < 2:
                # Try space-separated as fallback
                parts = line.split('  ', 1)
            if len(parts) < 2:
                logging.warning(f"Line {line_no+1}: cannot parse → '{line}'")
                continue

            normal_text = parts[0].strip()
            msl_gloss   = parts[1].strip()

         

## 6. Prepare data
Parse annotations.txt with the updated label logic, build the label vocabulary, and create train/val/test splits.

In [8]:
!bash scripts/01_prepare_data.sh

 Step 1 — Data Preparation
 Video dir       : data/videos
 Annotation file : data/annotations.txt

 Found 558 videos and 558 annotation lines

2026-08-09 03:49:53 | INFO     | prepare_data | ============================================================
2026-08-09 03:49:53 | INFO     | prepare_data | MSL Data Preparation
2026-08-09 03:49:53 | INFO     | prepare_data | ============================================================
2026-08-09 03:49:53 | INFO     | prepare_data | Parsing annotation file: data/annotations.txt
2026-08-09 03:49:53 | INFO     | prepare_data | Total annotation records: 558
2026-08-09 03:49:53 | INFO     | prepare_data | Vocabulary built: 541 unique classes
2026-08-09 03:49:53 | INFO     | prepare_data | Label map saved → data/label_map.json
2026-08-09 03:49:53 | INFO     | prepare_data | Matching videos from: data/videos
2026-08-09 03:49:53 | INFO     | prepare_data | NOTE: MSL4Emergency has ~1 video per sign class. The pipeline handles this via: (1) random (non-s

## 7. Diagnose and fix the mediapipe/protobuf conflict
Check installed protobuf and mediapipe versions, then upgrade protobuf to a version compatible with the rest of the environment. Verify with tools/probe_mediapipe.py that mediapipe imports correctly before retrying extraction.

In [11]:
!pip show protobuf mediapipe | grep -E "Name|Version"

Name: protobuf
Version: 5.29.5
Name: mediapipe
Version: 1.0.0


In [11]:
!pip install --upgrade "protobuf>=5.26,<6"
!python tools/probe_mediapipe.py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 6.8 MB/s eta 0:00:0000:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
Python  : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

[FAIL] Cannot import mediapipe at all: No module named 'mediapipe'


## 8. Extract keypoints
Run MediaPipe to extract pose and hand keypoints from all 558 videos, saving them as .npy files for training.

In [ ]:
!bash scripts/02_extract_keypoints.sh

In [24]:
!ls data/keypoints | wc -l
!cat data/keypoints/extraction_stats.json

1117
{
  "total": 558,
  "processed": 558,
  "failed": 0,
  "skipped": 0,
  "seq_len_min": 50,
  "seq_len_max": 387,
  "seq_len_mean": 130.56810035842295,
  "seq_len_std": 55.11924044288601,
  "seq_len_p50": 119.0,
  "seq_len_p95": 241.14999999999998,
  "lh_mean_presence": 0.6177208780096964,
  "rh_mean_presence": 0.7129535191739758
}

In [ ]:
!zip -r keypoints_backup.zip data/keypoints/

In [9]:
!ls -lh keypoints_backup.zip

-rw-r--r-- 1 root root 56M Aug  9 03:49 keypoints_backup.zip


## 9. Augment data
Apply 20x keypoint-space augmentation to the training split

In [ ]:
!bash scripts/03_augment_data.sh

In [25]:
!ls data/augmented | head -5
!python3 -c "import json; d=json.load(open('data/augmented/augmented_manifest.json')); print('train:', len(d['train']), '| val:', len(d['val']), '| test:', len(d['test']))"

augmented_manifest.json
train
val
train: 10602 | val: 558 | test: 558


## 10. Train, evaluate, and test BiLSTM
 
Train the BiLSTM model on the augmented keypoints using the column-2 MSL gloss labels. Evaluate the trained checkpoint on validation and test sets (val: 94.80% top-1, F1 93.13%; test: 100% — inflated due to train/test augmentation overlap from the dataset having ~1 video per class). Run inference on a sample video to confirm real-world predictions match the ground-truth gloss (idx20-244.mp4 correctly predicted as "လုံးဝတပ်မထား" at 56.4% confidence).

In [ ]:
!bash scripts/04_train.sh

In [18]:
!tail -30 results/exp_bilstm/logs/train_stdout.log

2026-08-07 10:58:38 | INFO     | train | Epoch  69/150 | train loss=1.0308 top1=99.74% | val loss=1.3047 top1=94.97% top5=96.53% | lr=3.34e-04 | 20s
2026-08-07 10:58:38 | INFO     | train |   ★ New best val top-1: 94.97%
2026-08-07 10:58:58 | INFO     | train | Epoch  70/150 | train loss=1.0308 top1=99.58% | val loss=1.3234 top1=94.10% top5=96.01% | lr=3.29e-04 | 20s
2026-08-07 10:59:18 | INFO     | train | Epoch  71/150 | train loss=1.0283 top1=99.68% | val loss=1.3032 top1=94.62% top5=96.18% | lr=3.24e-04 | 20s
2026-08-07 10:59:38 | INFO     | train | Epoch  72/150 | train loss=1.0263 top1=99.74% | val loss=1.3132 top1=94.44% top5=96.35% | lr=3.19e-04 | 20s
2026-08-07 10:59:58 | INFO     | train | Epoch  73/150 | train loss=1.0254 top1=99.73% | val loss=1.2973 top1=94.44% top5=96.35% | lr=3.14e-04 | 20s
2026-08-07 11:00:18 | INFO     | train | Epoch  74/150 | train loss=1.0255 top1=99.76% | val loss=1.3043 top1=94.79% top5=96.70% | lr=3.09e-04 | 20s
2026-08-07 11:00:38 | INFO     | t

In [10]:
!bash scripts/05_evaluate.sh bilstm exp_bilstm

 Evaluation — bilstm / exp_bilstm
 Checkpoint : results/exp_bilstm/checkpoints/best.pth
 Output dir : results/exp_bilstm/evaluation

── Validation set ──────────────────────────────────────────
2026-08-09 03:50:29 | INFO     | evaluate | Loaded checkpoint: results/exp_bilstm/checkpoints/best.pth  Model: bilstm
2026-08-09 03:50:29 | INFO     | evaluate | Evaluating 558 samples from 'val' split
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-08-09 03:50:30 | INFO     | evaluate | Model loaded. Evaluating 558 samples…
/usr/local/lib/python3.12/dist-packages/tor

In [15]:
!bash scripts/08_infer.sh video data/videos/idx20-244.mp4

 MSL Sign Language Inference
 Mode       : video
 Input      : data/videos/idx20-244.mp4
 Checkpoint : results/exp_bilstm/checkpoints/best.pth

/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.7.1, so it will not be used.
  warnings.warn(
2026-08-09 04:09:27 | INFO     | infer | Loaded bilstm from results/exp_bilstm/checkpoints/best.pth
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1786248568.151366     965 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786248568.194206     964 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786248568.253809     960 inference_feedback_manager.cc:114] Feedback manag

## 11. Train, evaluate, and test Transformer

Train the Transformer model on the augmented keypoints using the column-2 MSL gloss labels. Evaluate the trained checkpoint on validation and test sets (val: 97.13% top-1, F1 96.09%; test: 100% — same augmentation-overlap caveat as BiLSTM). Run inference on the same sample video (idx20-244.mp4) to compare against BiLSTM: Transformer also correctly predicted "လုံးဝတပ်မထား", though with lower confidence (31.3% vs BiLSTM's 56.4%).

In [16]:
!bash scripts/04_train.sh transformer exp_transformer

 Training MSL Recognition Model
 Model      : transformer
 Experiment : exp_transformer
 Config     : config/config.yaml

 Data: train=10602, val=558, test=558 (all-class design)

[GPU]
Tesla T4, 15360 MiB, 14909 MiB, 43
Tesla T4, 15360 MiB, 14909 MiB, 42

/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.7.1, so it will not be used.
  warnings.warn(
2026-08-09 04:13:35 | INFO     | train | Experiment: exp_transformer  Model: transformer
2026-08-09 04:13:35 | INFO     | train | Classes: 541
2026-08-09 04:13:35 | INFO     | train | Using all-class manifest: train=10602, val=558, test=558
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is go

In [22]:
!tail -30 results/exp_transformer/logs/train_stdout.log

2026-08-09 04:29:14 | INFO     | train | Epoch  79/150 | train loss=0.9871 top1=100.00% | val loss=1.1408 top1=97.22% top5=97.57% | lr=2.84e-04 | 12s
2026-08-09 04:29:14 | INFO     | train |   ★ New best val top-1: 97.22%
2026-08-09 04:29:26 | INFO     | train | Epoch  80/150 | train loss=0.9869 top1=100.00% | val loss=1.1418 top1=97.05% top5=97.74% | lr=2.78e-04 | 12s
2026-08-09 04:29:37 | INFO     | train | Epoch  81/150 | train loss=0.9932 top1=99.92% | val loss=1.1815 top1=96.53% top5=97.40% | lr=2.73e-04 | 12s
2026-08-09 04:29:49 | INFO     | train | Epoch  82/150 | train loss=1.0094 top1=99.63% | val loss=1.1694 top1=96.70% top5=97.57% | lr=2.68e-04 | 12s
2026-08-09 04:30:01 | INFO     | train | Epoch  83/150 | train loss=0.9918 top1=99.97% | val loss=1.1491 top1=97.22% top5=97.74% | lr=2.63e-04 | 12s
2026-08-09 04:30:13 | INFO     | train | Epoch  84/150 | train loss=0.9875 top1=100.00% | val loss=1.1517 top1=96.70% top5=97.40% | lr=2.58e-04 | 12s
2026-08-09 04:30:24 | INFO     

In [17]:
!bash scripts/05_evaluate.sh transformer exp_transformer

 Evaluation — transformer / exp_transformer
 Checkpoint : results/exp_transformer/checkpoints/best.pth
 Output dir : results/exp_transformer/evaluation

── Validation set ──────────────────────────────────────────
2026-08-09 04:34:42 | INFO     | evaluate | Loaded checkpoint: results/exp_transformer/checkpoints/best.pth  Model: transformer
2026-08-09 04:34:42 | INFO     | evaluate | Evaluating 558 samples from 'val' split
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-08-09 04:34:42 | INFO     | evaluate | Model loaded. Evaluating 558 samples…
/usr/local/li

In [18]:
!python src/infer.py --checkpoint results/exp_transformer/checkpoints/best.pth --config config/config.yaml --video data/videos/idx20-244.mp4 --top_k 5

/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.7.1, so it will not be used.
  warnings.warn(
2026-08-09 04:36:03 | INFO     | infer | Loaded transformer from results/exp_transformer/checkpoints/best.pth
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1786250163.974285    8226 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786250164.012543    8228 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786250164.059193    8224 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786250164.236012    8222 i

In [19]:
!find results/exp_transformer/checkpoints/ -type f ! -name "best.pth" -delete
!df -h /kaggle/working

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   15G  4.7G  77% /kaggle/working


## 12. Train, evaluate, and test ST-GCN

Train the ST-GCN model on the augmented keypoints using the column-2 MSL gloss labels (150 max epochs, patience=25). Training hit a display/kernel disconnect mid-run, but the log confirms it completed naturally via early stopping at epoch 82, with best val top-1 of 93.92% at epoch 57. Evaluate the best checkpoint on validation and test sets (val: 93.73% top-1, F1 92.22%; test: 100% — same augmentation-overlap caveat as the other two models). Run inference on the same sample video (idx20-244.mp4): ST-GCN also correctly predicted "လုံးဝတပ်မထား", with 32.5% confidence — consistent with BiLSTM (56.4%) and Transformer (31.3%) all agreeing on the same correct answer.

In [ ]:
!bash scripts/04_train.sh stgcn exp_stgcn

 Training MSL Recognition Model
 Model      : stgcn
 Experiment : exp_stgcn
 Config     : config/config.yaml

 Data: train=10602, val=558, test=558 (all-class design)

[GPU]
Tesla T4, 15360 MiB, 14909 MiB, 44
Tesla T4, 15360 MiB, 14909 MiB, 40

/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.7.1, so it will not be used.
  warnings.warn(
2026-08-09 04:40:11 | INFO     | train | Experiment: exp_stgcn  Model: stgcn
2026-08-09 04:40:11 | INFO     | train | Classes: 541
2026-08-09 04:40:11 | INFO     | train | Using all-class manifest: train=10602, val=558, test=558
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be

In [16]:
!tail -30 results/exp_stgcn/logs/train_stdout.log

2026-08-09 06:46:29 | INFO     | train | Epoch  57/150 | train loss=1.2058 top1=98.73% | val loss=1.3913 top1=93.92% top5=96.70% | lr=3.89e-04 | 132s
2026-08-09 06:46:29 | INFO     | train |   ★ New best val top-1: 93.92%
2026-08-09 06:48:41 | INFO     | train | Epoch  58/150 | train loss=1.1998 top1=98.75% | val loss=1.4156 top1=92.71% top5=95.83% | lr=3.85e-04 | 132s
2026-08-09 06:50:54 | INFO     | train | Epoch  59/150 | train loss=1.2020 top1=98.81% | val loss=1.4019 top1=92.88% top5=96.01% | lr=3.81e-04 | 132s
2026-08-09 06:53:06 | INFO     | train | Epoch  60/150 | train loss=1.1949 top1=98.95% | val loss=1.3907 top1=93.40% top5=96.35% | lr=3.76e-04 | 132s
2026-08-09 06:55:18 | INFO     | train | Epoch  61/150 | train loss=1.1927 top1=98.78% | val loss=1.4172 top1=92.88% top5=96.18% | lr=3.72e-04 | 132s
2026-08-09 06:57:30 | INFO     | train | Epoch  62/150 | train loss=1.1881 top1=98.84% | val loss=1.4000 top1=93.40% top5=95.66% | lr=3.67e-04 | 132s
2026-08-09 06:59:42 | INFO  

In [15]:
!ls -la results/exp_stgcn/checkpoints/

total 1250604
drwxr-xr-x 2 root root     4096 Aug  9 07:59 .
drwxr-xr-x 5 root root     4096 Aug  9 08:03 ..
-rw-r--r-- 1 root root 33694024 Aug  9 07:59 best.pth
-rw-r--r-- 1 root root 33694024 Aug  9 08:00 epoch020_val0.8889.pth
-rw-r--r-- 1 root root 33694024 Aug  9 07:59 epoch026_val0.9149.pth
-rw-r--r-- 1 root root 33694024 Aug  9 07:59 epoch027_val0.9097.pth
-rw-r--r-- 1 root root 33694024 Aug  9 08:00 epoch028_val0.9132.pth
-rw-r--r-- 1 root root 33694024 Aug  9 08:00 epoch031_val0.9149.pth
-rw-r--r-- 1 root root 33694024 Aug  9 07:59 epoch036_val0.9219.pth
-rw-r--r-- 1 root root 33694024 Aug  9 08:00 epoch037_val0.9219.pth
-rw-r--r-- 1 root root 33694024 Aug  9 08:00 epoch039_val0.9253.pth
-rw-r--r-- 1 root root 33694024 Aug  9 08:00 epoch041_val0.9253.pth
-rw-r--r-- 1 root root 33694024 Aug  9 08:00 epoch044_val0.9271.pth
-rw-r--r-- 1 root root 33694024 Aug  9 07:59 epoch047_val0.9288.pth
-rw-r--r-- 1 root root 33694024 Aug  9 08:00 epoch050_val0.9323.pth
-rw-r--r-- 1 root roo

In [6]:
!bash scripts/05_evaluate.sh stgcn exp_stgcn

 Evaluation — stgcn / exp_stgcn
 Checkpoint : results/exp_stgcn/checkpoints/best.pth
 Output dir : results/exp_stgcn/evaluation

── Validation set ──────────────────────────────────────────
2026-08-09 08:05:45 | INFO     | evaluate | Loaded checkpoint: results/exp_stgcn/checkpoints/best.pth  Model: stgcn
2026-08-09 08:05:46 | INFO     | evaluate | Evaluating 558 samples from 'val' split
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-08-09 08:05:46 | INFO     | evaluate | Model loaded. Evaluating 558 samples…
/usr/local/lib/python3.12/dist-packages/torch/uti

In [21]:
!python src/infer.py --checkpoint results/exp_stgcn/checkpoints/best.pth --config config/config.yaml --video data/videos/idx20-244.mp4 --top_k 5

/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.7.1, so it will not be used.
  warnings.warn(
2026-08-09 08:19:36 | INFO     | infer | Loaded stgcn from results/exp_stgcn/checkpoints/best.pth
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1786263576.728614    2047 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786263576.766250    2047 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786263576.826064    2042 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786263577.001597    2042 inference_fee

## 13. Compare all three models


| Metric | BiLSTM | Transformer | ST-GCN |
|---|---|---|---|
| Val accuracy | 94.80% | 97.13% | 93.73% |
| Val F1 | 93.13% | 96.09% | 92.22% |
| Test video prediction | Correct (56.4%) | Correct (31.3%) | Correct (32.5%) |

Transformer performed best overall — highest accuracy and fastest to train. BiLSTM was in the middle. ST-GCN was slowest to train and slightly less accurate than the other two.

All three models correctly predicted the same test video using the MSL gloss (column 2) labels, so the column-2 approach works well across different model types.

Note: test accuracy for all models showed 100%, but this is misleading — it's because the dataset only has ~1 video per class, so augmented copies of the test videos are already in the training set. Validation accuracy is the fair number to compare.